[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/magnetic_resonance/magnetic_resonance_space.ipynb)

# Magnetic Resonance in the Algebra of Space

This example computes magnetic resonance, following driven spin nutation, power-broadened absorption and dispersion spectra, and Hahn spin-echo refocusing in an inhomogeneous field. Spins placed in a static magnetic field precess in the field's plane, from electron spins in solid-state resonance to nuclear spins in spectroscopy and magnetic resonance imaging. A radio-frequency field tuned near the precession frequency tips the spins away from equilibrium, while coupling to the surrounding bath pulls longitudinal magnetization back toward the static field and dephases transverse coherence.

In the algebra of three-dimensional space, an ensemble spin state is modelled as `0.5 * (one + bloch)`, a scalar plus an extrinsic spatial Bloch vector, with the spatial `pseudoscalar` supplying the unit of phase in `-2 * pseudoscalar * hamiltonian.commutator(State)` and Lindblad jump sandwiches `process * State * ~process` driving longitudinal and transverse relaxation. Because `pseudoscalar` fixes a spatial handedness that physical spins do not carry, reflecting the setup in a mirror runs precession backwards in this representation. Both the generator and its short-time exponential remain linear maps on `State`, so composing doubled waiting maps with pulse sandwiches `waiting(turn(waiting(tip)))` builds the whole spin-echo experiment as one ensemble-averaged map. The [spacetime twin notebook](magnetic_resonance.ipynb) models each spin intrinsically as the patch of its turning plane in spacetime, avoiding any chosen spatial handedness.

In matrix Lindblad notation, the state reads as a $2 \times 2$ Hermitian density matrix $\rho = \tfrac12(I + \mathbf{r}\cdot\boldsymbol{\sigma})$, reversion as Hermitian conjugation, the unit volume element as $i$, and the generator as the $4 \times 4$ superoperator $\dot{\rho} = -i[H, \rho] + \sum_k \big(L_k \rho L_k^\dagger - \tfrac12\{L_k^\dagger L_k, \rho\}\big)$, whose vector part gives the Bloch equations $\dot{\mathbf{r}} = \boldsymbol{\omega} \times \mathbf{r} - (r_x / T_2,\, r_y / T_2,\, (r_z - 1) / T_1)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Generator, Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.quantum.magnetic_resonance import render
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()                                         # Vector+
# Self-reverse elements combine a scalar trace with a spatial Bloch vector.
State = ga.gatype.self_reverse()                                    # Scalar+ + Vector+
Rates = ga.gatype((State, State))                                   # State+ <- State+
Evolution = ga.gatype((State, State))                               # State+ <- State+
# Lindblad jump operators span spatial vectors and bivectors.
Process = ga.gatype.from_grades([1, 2])                             # Vector+ + Bivector+
Rotor = ga.gatype.rotor()                                           # Rotor+
pseudoscalar = mv.xyz                                               # [] Pseudoscalar+
one = mv.scalar([1])                                                # [] Scalar+
# Relaxation times in microseconds, with angular frequencies in radians per microsecond.
t1, t2 = 10.0, 4.0

## Spin states and the rate generator

An ensemble spin state in three-dimensional space combines half of `one` with half of the Bloch vector `bloch`, which has unit length for a pure state and shorter length for a mixture. Precession in the frame rotating with the drive acts through `-2 * pseudoscalar * hamiltonian.commutator(State)` around the detuning along `z` and the drive along `x`. Relaxation acts through Lindblad processes whose sandwiches `process * State * ~process` move population and phase, where `raise_process` flips states opposed to the field back onto `z` at rate `1 / t1` and a half turn in `xy` damps transverse phase coherence toward `1 / t2`. Together precession and relaxation form a single linear map `rate_map` over `State`.

In [ ]:
detuning, drive = 0.7, 1.3
sample_bloch = mv.vector([0.4, -0.2, 0.6])                                      # [] Vector+

In [ ]:
def spin_state(bloch: Vector) -> State:
    """Forms the spatial spin state `0.5 * (one + bloch)` (`[...] State+`) of `bloch`."""
    return 0.5 * (one + bloch)                                                  # [...] State+


def relaxation_map(process: Process) -> Rates:
    """Forms the Lindblad dissipator (`[] State+ <- State+`) generated by `process`."""
    back = process.reverse().symmetric_reverse_product()                        # [] State+
    # Multiplying a self-reverse state on the left by `process` and on the right by `~process` preserves self-reversion.
    jump = (process * State * ~process).cast(Rates)                             # [] State+ <- State+
    return jump - back.anticommutator(State)                                    # [] State+ <- State+


raise_process = mv.x * spin_state(-mv.z)                                        # [] Process+

In [ ]:
def rate_map(detuning: np.ndarray, drive: np.ndarray, t1: float, t2: float) -> Rates:
    """Combines rotating-frame precession with longitudinal `1 / t1` and transverse `1 / t2`
    Lindblad relaxation into one generator (`[...] State+ <- State+`)."""
    hamiltonian = 0.5 * (mv.z * detuning + mv.x * drive)                        # [...] Vector+
    turning = -2 * pseudoscalar * hamiltonian.commutator(State)                 # [...] Vector+ <- State+
    relaxing = (1 / t1) * relaxation_map(raise_process) + 0.5 * (1 / t2 - 1 / (2 * t1)) * relaxation_map(mv.xy)  # [] State+ <- State+
    return turning + relaxing                                                   # [...] State+ <- State+


sample_rates = rate_map(detuning, drive, t1, t2)                                # [] State+ <- State+
sample_change = sample_rates(spin_state(sample_bloch))                          # [] State+
print("rate of change:\n", sample_change.coefficient_table())

## Driven nutation and the steady state

Under a continuous radio-frequency drive, each spin nutates around the effective rotating-frame field while relaxation damps the trajectory into a stationary state where the generator vanishes. Because `rate_map` preserves the scalar trace of every state, pinning that scalar part to one half makes the operator invertible so a single linear solve determines `steady_state`. Repeatedly applying the fourth-order propagator `step_evolution` traces each spin's spiral from the north pole into its solved stationary point.

In matrix superoperator notation, short-time evolution over $dt$ reads as the matrix exponential $e^{\mathcal{L}\,dt}$, whose fourth-order series truncation matches the classical fourth-order Runge-Kutta step of a time-independent linear equation.

In [ ]:
nutation_detunings = np.array([0.0, 0.5, 1.0])
nutation_drive = 1.0
nutation_dt = 0.02
n_nutation_steps, nutation_stride = 2000, 5

In [ ]:
def steady_state(rates: Rates) -> State:
    """Solves for the stationary state (`[...] State+`) of `rates` by pinning the scalar trace to one half."""
    scalar_part = one.scalar_product(State)                                     # [] Scalar+ <- State+
    return (rates + one * scalar_part).solve(one * 0.5)                         # [...] State+


def step_evolution(rates: Rates, dt: float) -> Evolution:
    """Approximates the short-time propagator over `dt` as the fourth-order composition series of
    `rates * dt` (`[...] State+ <- State+`)."""
    exponent = rates * dt                                                       # [...] State+ <- State+
    term = total = State
    for order in range(1, 5):
        # Each power in the exponential series composes the previous map with `exponent`.
        term = exponent(term) / order                                           # [...] State+ <- State+
        total = total + term
    return total                                                                # [...] State+ <- State+


driven_rates = rate_map(nutation_detunings, nutation_drive, t1, t2)             # [3] State+ <- State+
steady_states = steady_state(driven_rates)                                      # [3] State+
nutation_step = step_evolution(driven_rates, nutation_dt)                       # [3] State+ <- State+

In [ ]:
def trajectory(step_map: Evolution, state: State, n_steps: int) -> Generator[State, None, State]:
    """Yields `state` (`[...] State+`) before each of `n_steps` applications of `step_map` and returns
    the state after the last step."""
    for _ in range(n_steps):
        yield state                                                             # [...] State+
        state = step_map(state)
    return state


equilibrium = spin_state(mv.z)                                                  # [] State+
nutation_start = equilibrium.broadcast_to(nutation_detunings.shape)             # [3] State+
nutation_states = list(trajectory(nutation_step, nutation_start, n_nutation_steps))
save_figure(render.nutation_figure(nutation_states[::nutation_stride], steady_states), "resonance_nutation_space");

All trajectories start at thermal equilibrium along `z` at the top of the Bloch ball and spiral into their solved stationary dots. On resonance (red) the Bloch vector nutates purely in the `yz` plane perpendicular to the `x` drive axis, whereas larger detunings (purple and blue) tilt the precession cone toward `z` and settle closer to equilibrium.

## Power-broadened spectral lines

Steady-state absorption measures the Bloch component lagging a quarter turn behind the drive along `-y`, while dispersion measures the in-phase component along `x`. Because `steady_state` reduces stationary balance to a linear solve, evaluating an entire detuning sweep across weak, intermediate, and saturating drives takes one broadcast call.

In the component form of the Bloch equations, the stationary Bloch vector reads $r_z = (1 + \Delta^2 T_2^2) / d$, $-r_y = \Omega T_2 / d$, and $r_x = \Delta \Omega T_2^2 / d$ with $d = 1 + \Delta^2 T_2^2 + \Omega^2 T_1 T_2$, giving a power-broadened half-width $\sqrt{1 + \Omega^2 T_1 T_2} / T_2$.

In [ ]:
n_line_detunings = 241
line_detunings = np.linspace(-3.0, 3.0, n_line_detunings)
line_drives = np.array([0.05, 0.3, 1.0])

In [ ]:
line_rates = rate_map(line_detunings[:, None], line_drives[None, :], t1, t2)    # [n_line_detunings, 3] State+ <- State+
line_states = steady_state(line_rates)                                          # [n_line_detunings, 3] State+
save_figure(render.lines_figure(line_detunings, line_drives, line_states), "resonance_lines_space");

Normalized per unit drive, the weakest drive (red) traces a tall, narrow Lorentzian absorption peak and sharp antisymmetric dispersion wings governed by transverse dephasing alone. Stronger drives (purple and blue) saturate the longitudinal population difference, lowering peak absorption per unit drive and broadening both curves across detuning.

## Spin echo

Static field inhomogeneity gives each spin in a sample a slightly different detuning, causing an ensemble tipped into the transverse plane to fan out and lose its mean pickup-coil signal long before irreversible dephasing acts. Applying a resonant half-turn pulse about `x` after a delay flips every Bloch vector across the drive axis so faster spins trail slower ones and rephase into a Hahn echo at twice the delay. Each short pulse is a rotor `(pseudoscalar * axis * (-angle / 2)).exp()` acting on `State` by its sandwich.

In NMR notation, static inhomogeneous dephasing cancels at $t = 2\tau$, leaving an echo envelope $e^{-2\tau / T_2}$.

In [ ]:
n_spins, spread, seed = 300, 2.0, 0
delay, echo_dt = 2.5, 0.01
n_echo_steps = 800
frame_stride, frame_duration_ms = 8, 40

In [ ]:
def drive_pulse(angle: float) -> Rotor:
    """Forms the rotor (`[] Rotor+`) of a resonant pulse turning the Bloch vector about `x` by `angle`."""
    return (pseudoscalar * mv.x * (-angle / 2)).exp()                           # [] Rotor+


def echo_trajectory(step_map: Evolution, state: State, n_before: int, n_after: int) -> Iterator[State]:
    """Yields states (`[...] State+`) through a Hahn spin echo: a quarter-turn pulse about `x`, `n_before`
    free steps, a half-turn refocusing pulse, and `n_after` free steps."""
    tipped = drive_pulse(np.pi / 2) >> state                                    # [...] State+
    state = yield from trajectory(step_map, tipped, n_before)
    turned = drive_pulse(np.pi) >> state                                        # [...] State+
    yield from trajectory(step_map, turned, n_after)


spin_detunings = np.random.default_rng(seed).normal(0.0, spread, n_spins)
free_rates = rate_map(spin_detunings, 0.0, t1, t2)                              # [n_spins] State+ <- State+
echo_start = equilibrium.broadcast_to(spin_detunings.shape)                     # [n_spins] State+
n_delay_steps = round(delay / echo_dt)
echo_stream = echo_trajectory(
    step_evolution(free_rates, echo_dt), echo_start, n_delay_steps, n_echo_steps - n_delay_steps
)
echo_ensemble = stack(
    show_states(
        echo_stream,
        render.echo_frames,
        spin_detunings,
        echo_dt,
        delay,
        t2,
        frame_stride,
        name="resonance_echo_space",
        duration_ms=frame_duration_ms,
    )
)                                                                               # [n_echo_steps, n_spins] State+

Viewed down the static field axis on the left, Bloch vectors coloured from negative (blue) to positive (red) detuning fan around the transverse circle after the initial quarter-turn pulse, flip across the horizontal drive axis at the half-turn pulse, and reconverge at twice the pulse delay. On the right, the mean transverse signal (blue) collapses during the initial fan-out and rises back to the dashed `t2` relaxation envelope at the echo peak before dephasing again.

## Pulse sequences as composed maps

Because `step_evolution` is a linear map on `State`, composing a span with itself `span(span)` doubles the waiting interval at each iteration, reaching a thousand time steps in ten compositions. Turning the quarter-turn and half-turn pulse rotors into state maps `pulse_rotor >> State` lets the entire Hahn echo `waiting(turn(waiting(tip)))` and the unrefocused free induction decay `waiting(waiting(tip))` be assembled and averaged across the inhomogeneous ensemble before any initial state is supplied.

In Liouville superoperator notation, the ensemble-averaged echo propagator reads as $\langle e^{\mathcal{L}\tau}\,\mathcal{R}_\pi\,e^{\mathcal{L}\tau}\,\mathcal{R}_{\pi/2}\rangle$, whose transverse echo amplitude decays as $e^{-t / T_2}$ across total elapsed time $t = 2\tau$.

In [ ]:
n_doublings = 11

In [ ]:
def doubled_spans(span: Evolution, n_doublings: int) -> Iterator[Evolution]:
    """Yields `n_doublings` evolution maps (`[...] State+ <- State+`), composing each span with itself
    to double the elapsed time."""
    for _ in range(n_doublings):
        yield span                                                              # [...] State+ <- State+
        span = span(span)


waiting = stack(list(doubled_spans(step_evolution(free_rates, echo_dt), n_doublings)))  # [n_doublings, n_spins] State+ <- State+
times = 2 * echo_dt * 2.0 ** np.arange(n_doublings)
tip = drive_pulse(np.pi / 2) >> State                                           # [] State+ <- State+
turn = drive_pulse(np.pi) >> State                                              # [] State+ <- State+
echo_map = waiting(turn(waiting(tip))).mean(axis=-1)                            # [n_doublings] State+ <- State+
decay_map = waiting(waiting(tip)).mean(axis=-1)                                 # [n_doublings] State+ <- State+
echo_states = echo_map(equilibrium)                                             # [n_doublings] State+
decay_states = decay_map(equilibrium)                                           # [n_doublings] State+
save_figure(render.decays_figure(times, echo_states, decay_states, t2), "resonance_echo_decay_space");

Across three decades of total delay, the unrefocused free decay (red) drops to zero within a microsecond as static detuning spreads the ensemble around the equator, whereas the refocused Hahn echo amplitudes (blue) follow the dashed `t2` curve set by irreversible transverse relaxation.

In [ ]:
# checks
axes = stack((mv.x, mv.y, mv.z))                                                # [3] Vector+
sample_readings = np.array([0.4, -0.2, 0.6])
expected_rates = np.cross([drive, 0.0, detuning], sample_readings) - np.array(
    [sample_readings[0] / t2, sample_readings[1] / t2, (sample_readings[2] - 1) / t1]
)
assert (((2 * sample_change[..., None]).scalar_product(axes) - expected_rates).abs() < 1e-10).all()
assert ((nutation_states[-1] - steady_states).scalar_norm_squared() < 1e-4).all()
assert (driven_rates(steady_states).scalar_norm_squared() < 1e-20).all()
assert (line_rates(line_states).scalar_norm_squared() < 1e-20).all()

grid_detunings, grid_drives = line_detunings[:, None], line_drives[None, :]
denominator = 1 + (grid_detunings * t2) ** 2 + grid_drives**2 * t1 * t2
line_vectors = 2 * line_states.select.vector()                                  # [n_line_detunings, 3] Vector+
assert (((mv.z | line_vectors) - (1 + (grid_detunings * t2) ** 2) / denominator).abs() < 1e-10).all()
assert ((-(mv.y | line_vectors) - grid_drives * t2 / denominator).abs() < 1e-10).all()
assert (((mv.x | line_vectors) - grid_detunings * grid_drives * t2**2 / denominator).abs() < 1e-10).all()

refocused_mean = echo_ensemble[2 * n_delay_steps].mean(axis=-1).select.vector()  # [] Vector+
refocused_transverse = (refocused_mean - (mv.xy >> refocused_mean)).norm()       # [] Scalar+
assert ((refocused_transverse - np.exp(-2 * delay / t2)).abs() < 0.02).all()

echo_vectors = echo_states.select.vector()                                      # [n_doublings] Vector+
echo_transverse = (echo_vectors - (mv.xy >> echo_vectors)).norm()               # [n_doublings] Scalar+
assert ((echo_transverse - np.exp(-times / t2)).abs() < 1e-5).all()